In [ ]:
%pip install pandas numpy matplotlib seaborn geopandas shapely statsmodels scikit-learn

import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely import wkt
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
import statsmodels.api as sm
from sklearn.tree import plot_tree
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import statsmodels.api as sm

In [ ]:
ridership = pd.read_csv("resources/ridership_by_station.csv", dtype={'station_id': str})
ridership.head()

In [ ]:
ridership['date'] = pd.to_datetime(ridership['date'])
ridership['daytype'] = ridership['daytype'].astype('category')
ridership['rides'] = pd.to_numeric(ridership['rides'], errors='coerce')
ridership['station_id'] = pd.to_numeric(ridership['station_id'], errors='coerce')
ridership.info()

In [ ]:
ridership.head()

In [ ]:
#Import weather data
with open('weather_data.pkl', 'rb') as f:
    weather = pickle.load(f)
weather.head()

In [ ]:
#Add weather data to station ridership data
ridership = pd.merge(ridership, weather, left_on='date', right_on='DATE', how='left')
ridership.drop(inplace=True, columns=['STATION', 'NAME', 'DATE'])
display(ridership)

In [ ]:
station_info = pd.read_csv("resources/system_info.csv")
station_info.head()

In [ ]:
station_info['DIRECTION_ID'] = station_info['DIRECTION_ID'].astype('category')
station_info['MAP_ID'] = station_info['MAP_ID'].astype(int)
for col in station_info.select_dtypes(include=['bool']).columns:
    station_info[col] = station_info[col].astype(int)

station_info[['latitude', 'longitude']] = station_info['Location'].str.strip().str.strip('()').str.split(', ', expand=True)
station_info['latitude'] = pd.to_numeric(station_info['latitude'].str.strip())
station_info['longitude'] = pd.to_numeric(station_info['longitude'].str.strip())
station_info = station_info[['STATION_NAME', 'DIRECTION_ID', 'MAP_ID', 'ADA', 'RED', 'BLUE', 'G', 'BRN', 'P', 'Y', 'Pnk', 'O', 'latitude', 'longitude']]
station_info.head()

In [ ]:
#Identify which stations have conflicting ADA status (i.e. part of station or certain lines are ADA compliant while others aren't)
grouped_ada = station_info.groupby('MAP_ID')['ADA'].apply(lambda x: x.unique())
conflicting_stations = grouped_ada[grouped_ada.apply(lambda x: 0 in x and 1 in x)].index.tolist()

print("STATION_NAMEs with conflicting ADA values:", conflicting_stations)

In [ ]:
agg_dict = {
    'STATION_NAME': 'first',
    'latitude': 'first',
    'longitude': 'first',
    'DIRECTION_ID': lambda x: tuple(x.unique()),
    'ADA': 'max',
    'RED': 'max',
    'BLUE': 'max',
    'G': 'max',
    'BRN': 'max',
    'P': 'max',
    'Y': 'max',
    'Pnk': 'max',
    'O': 'max'
}

si_aggregated = station_info.groupby('MAP_ID').agg(agg_dict).reset_index()
si_aggregated.head()

In [ ]:
#Turn direction into a dummy variable
direction_dummies = si_aggregated['DIRECTION_ID'].apply(lambda x: '|'.join(map(str, x))).str.get_dummies()
si_aggregated = si_aggregated.join(direction_dummies).drop(columns=['DIRECTION_ID'])
si_aggregated['N/S'] = si_aggregated['N'] | si_aggregated['S']
si_aggregated['E/W'] = si_aggregated['E'] | si_aggregated['W']
si_aggregated = si_aggregated.drop(columns=['N', 'S', 'E', 'W'])
si_aggregated.head()

In [ ]:
station_ridership = pd.merge(ridership, si_aggregated, left_on='station_id', right_on='MAP_ID', how='left')
display(station_ridership)

In [ ]:
start_date = pd.to_datetime('2000-01-01')
station_ridership['days_since_2000'] = (station_ridership['date'] - start_date).dt.days

#Calculate month (as integer)
station_ridership['month'] = station_ridership['date'].dt.month

#Calculate year
station_ridership['year'] = station_ridership['date'].dt.year

#Define a function to get the season
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

#Apply the function to get the season column
station_ridership['season'] = station_ridership['month'].apply(get_season)

#Display the first few rows with the new columns
print(station_ridership[['date', 'days_since_2000', 'season', 'month', 'year']].head())

In [ ]:
#Add season as dummy variable
season_dummies = pd.get_dummies(station_ridership['season'], prefix='season', drop_first=True)
station_ridership = pd.concat([station_ridership, season_dummies], axis=1)
season_dummies = season_dummies.astype(int)
station_ridership.head()
station_ridership.columns

In [ ]:
#Add variable that represents days since initial school shutdown in Chicago for COVID
shutdown_date = pd.to_datetime('2020-03-13')
station_ridership['post_shutdown'] = (station_ridership['date'] >= shutdown_date).astype(int)
station_ridership.head()

In [ ]:
#Identify rows with missing latitude or longitude
missing_coords_df = station_ridership[station_ridership['latitude'].isnull() | station_ridership['longitude'].isnull()]

#Get unique station names with missing coordinates, excluding NaN values
unique_stations_missing_coords = missing_coords_df['stationname'].dropna().unique()

if len(unique_stations_missing_coords) == 0:
    print("No stations found with missing latitude or longitude values.")
else:
    print("Stations with missing latitude or longitude values and their date range:")
    for station_name in unique_stations_missing_coords:
        # Filter for the current station and missing coordinates using 'stationname'
        station_missing_data = missing_coords_df[missing_coords_df['stationname'] == station_name]

        # Get the first and last date for this station's missing coordinates
        first_date = station_missing_data['date'].min()
        last_date = station_missing_data['date'].max()

        print(f"{station_name}, {first_date.strftime('%Y-%m-%d')} through {last_date.strftime('%Y-%m-%d')}")

In [ ]:
columns_for_splom = ['days_since_2000', 'post_shutdown', 'month', 'year', 'AWND', 'PRCP',
       'SNOW', 'SNWD', 'TAVG', 'station_id', 'N/S', 'E/W', 'RED', 'BLUE', 'G', 'BRN', 'P', 'Y', 'Pnk', 'O', 'latitude', 'longitude', 'ADA', 'rides']

#See correlations
plt.figure(figsize=(12,10))
!sns.heatmap(station_ridership[columns_for_splom].corr(), annot=True, cmap='coolwarm')

In [ ]:
!sns.pairplot(station_ridership, x_vars=['days_since_2000', 'month', 'year', 'station_id', 'N/S', 'E/W', 'RED', 'BLUE', 'G', 'BRN', 'P', 'Y', 'Pnk', 'O', 'latitude', 'longitude', 'ADA'], y_vars=['rides'])
!plt.show()

In [ ]:
!sns.pairplot(station_ridership[['days_since_2000', 'month', 'year', 'station_id', 'latitude', 'longitude', 'rides']])
!plt.show()

In [ ]:
zoning_df = pd.read_csv("resources/zoning_districts_2026.csv")

# Convert WKT strings to actual geometry objects
zoning_df['geometry'] = zoning_df['the_geom'].apply(wkt.loads)
zoning_df['ZONE_TYPE'] = zoning_df['ZONE_CLASS'].apply(lambda x: re.split(r'[-\s]+', x)[0])
# Now create the GeoDataFrame with the geometry column
zoning_gdf = gpd.GeoDataFrame(zoning_df, geometry='geometry', crs='EPSG:4326')

zone_classes = sorted(zoning_gdf['ZONE_TYPE'].unique())
for zone_class in zone_classes:
    print(f"CLASS: {zone_class}")

print(zoning_gdf['geometry'])

In [ ]:
zone_translation_dict = {
    'B1': 'Neighborhood Shopping District',
    'B2': 'Neighborhood Mixed-Use District',
    'B3': 'Community Shopping District',
    'C1': 'Neighborhood Commercial District',
    'C2': 'Motor Vehicle-Related Commercial District',
    'C3': 'Commercial, Manufacturing and Employment District',
    'DC': 'Downtown Core District',
    'DR': 'Downtown Residential District',
    'DS': 'Downtown Service District',
    'DX': 'Downtown Mixed-Use District',
    'M1': 'Limited Manufacturing/Business Park District',
    'M2': 'Light Industry District',
    'M3': 'Heavy Industry District',
    'PD': 'Planned Developments',
    'PMD': 'Planned Manufacturing District',
    'PMD13': 'Planned Manufacturing District',
    'POS': 'Regional or Community Park',
    'RM': 'Residential Multi-Unit District',
    'RM4': 'Residential Multi-Unit District',
    'RM4.5': 'Residential Multi-Unit District',
    'RM5.5': 'Residential Multi-Unit District',
    'RS': 'Residential Single-Unit District',
    'RT': 'Residential Two-Flat, Townhouse and Multi-Unit District',
    'T': 'Transportation District'
}

zone_translation_column_dict = zone_translation_dict.copy()
for key in zone_translation_column_dict.keys():
    zone_translation_column_dict[key] = re.sub(r'[^a-zA-Z0-9]+', '_', zone_translation_column_dict[key]).lower()

zone_translation_column_dict

In [ ]:
# Plot the zoning districts
fig, ax = plt.subplots(figsize=(15, 15))
zoning_gdf.plot(ax=ax, column='ZONE_TYPE', legend=True, cmap='tab20', alpha=0.6, edgecolor='black', linewidth=0.5)

# Add station locations
ax.scatter(si_aggregated['longitude'], si_aggregated['latitude'], 
           c='red', s=50, alpha=0.7, edgecolors='darkred', linewidth=1, 
           label='CTA Stations', zorder=5)

ax.set_title('Chicago Zoning Districts by Class', fontsize=16)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
si_aggregated

In [ ]:
# Create a GeoDataFrame with station locations with shapes representing a 1 mile buffer around each station
si_gdf = gpd.GeoDataFrame(
    si_aggregated,
    geometry=gpd.points_from_xy(si_aggregated['longitude'], si_aggregated['latitude']),
    crs='EPSG:4326'
)

# Project to a projected CRS that uses meters (Chicago area: NAD83 / Illinois East)
si_gdf_projected = si_gdf.to_crs('EPSG:2790')

# Create 1-mile buffers (1 mile = 1609.34 meters)
feet_per_mile = 5280
meters_per_foot = 0.3048
meters_per_mile = feet_per_mile * meters_per_foot  # 1609.344 meters

si_gdf_projected['geometry_mile'] = si_gdf_projected.geometry.buffer(meters_per_mile)
si_gdf_projected['geometry_half_mile'] = si_gdf_projected.geometry.buffer(meters_per_mile / 2)

# WGS84 is EPSG:4326, where coordinates are in decimal degrees
# (x = longitude, y = latitude), not linear units like meters/feet.
si_gdf['geometry_mile'] = gpd.GeoSeries(
    si_gdf_projected['geometry_mile'], crs=si_gdf_projected.crs
).to_crs(si_gdf.crs)

si_gdf['geometry_half_mile'] = gpd.GeoSeries(
    si_gdf_projected['geometry_half_mile'], crs=si_gdf_projected.crs
).to_crs(si_gdf.crs)
display(si_gdf.head())

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))
zoning_gdf.plot(ax=ax, column='ZONE_TYPE', legend=True, cmap='tab20', alpha=0.6, edgecolor='black', linewidth=0.5)

# Plot station points
ax.scatter(si_gdf['longitude'], si_gdf['latitude'], 
           c='darkred', s=100, alpha=0.8, edgecolors='black', linewidth=1, 
           label='CTA Stations', zorder=5)

ax.set_title('Chicago CTA Stations with 1-Mile Service Areas', fontsize=16)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()


si_gdf['geometry_mile'].plot(
    ax=ax,
    facecolor='none',
    edgecolor='red',
    alpha=0.2,
    linewidth=1.5,
    zorder=4
)

si_gdf['geometry_half_mile'].plot(
    ax=ax,
    facecolor='none',
    edgecolor='blue',
    alpha=0.2,
    linewidth=1.5,
    zorder=4
)

# ax.set_ylim(miny, maxy)
# minx, miny, maxx, maxy = zoning_gdf.total_bounds
# ax.set_xlim(minx, maxx)

plt.show()

In [ ]:

for zone_class in zone_classes:
    zone_polygons = zoning_gdf[zoning_gdf['ZONE_TYPE'] == zone_class]['geometry']
    zone_class_column_name = zone_translation_column_dict[zone_class]

    mile_column_name = f"{zone_class_column_name}_within_mile"
    half_mile_column_name = f"{zone_class_column_name}_within_half_mile"
    
    si_gdf[mile_column_name] = si_gdf['geometry_mile'].apply(lambda x: zone_polygons.intersects(x).any()).astype(int)
    si_gdf[half_mile_column_name] = si_gdf['geometry_half_mile'].apply(lambda x: zone_polygons.intersects(x).any()).astype(int)

si_gdf.head()

In [ ]:
ridership_with_zoning = pd.merge(station_ridership, si_gdf.drop(columns=['geometry', 'geometry_mile', 'geometry_half_mile']), left_on='MAP_ID', right_on='MAP_ID', how='left')

# ridership_with_zoning.head()
# Remove duplicate columns from merge: keep left-table versions (_x), drop right-table duplicates (_y)
y_cols = [c for c in ridership_with_zoning.columns if c.endswith('_y')]
x_cols = [c for c in ridership_with_zoning.columns if c.endswith('_x')]

ridership_with_zoning = ridership_with_zoning.drop(columns=y_cols)
ridership_with_zoning = ridership_with_zoning.rename(columns={c: c[:-2] for c in x_cols})
ridership_with_zoning.columns

In [ ]:
len(ridership_with_zoning.columns)

In [ ]:
# Build a modeling dataset from ridership_with_zoning
model_df = ridership_with_zoning.copy()

# Keep core temporal/transit features + zoning proximity flags
base_features = [
    'days_since_2000', 'post_shutdown', 'season_Spring', 'season_Summer', 'season_Winter',
    'year', 'PRCP', 'TAVG', 'SNOW', 'SNWD', 'Fog_Ice_Fog', 'Blowing_Snow',
    'latitude', 'longitude', 'ADA', 'RED', 'BLUE', 'G', 'BRN', 'P', 'Y', 'Pnk', 'O', 'N/S', 'E/W'
]
zoning_features = [c for c in model_df.columns if c.endswith('_mile') or c.endswith('_half_mile')]
feature_cols = [c for c in base_features if c in model_df.columns] + zoning_features

X = model_df[feature_cols].copy()
y = model_df['rides'].copy()

# Add daytype as dummy variables
daytype_dummies = pd.get_dummies(model_df['daytype'], prefix='daytype', drop_first=True, dtype=int)
X = pd.concat([X, daytype_dummies], axis=1)

# Convert boolean columns to int
bool_cols = X.select_dtypes(include='bool').columns
X[bool_cols] = X[bool_cols].astype(int)

# Force all predictors/target to numeric for statsmodels compatibility
X = X.apply(pd.to_numeric, errors='coerce')
y = pd.to_numeric(y, errors='coerce')

# Drop rows with any missing values in features/target
valid_mask = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_mask]
y = y.loc[valid_mask]

# Optional sampling for speed on very large dataset
sample_n = min(200_000, len(X))
sample_idx = X.sample(n=sample_n, random_state=42).index
#X = X.loc[sample_idx]
#y = y.loc[sample_idx]

In [ ]:
X.head()

In [ ]:
# 1. Define your cutoff (approximate days from 2000 to end of 2023)
# 24 years * 365.25 days = 8766
#cutoff_value = 8766
cutoff_value = 25*365.25

# 2. Create the mask
train_mask = X['days_since_2000'] <= cutoff_value
test_mask = X['days_since_2000'] > cutoff_value

# 3. Split the data
X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]

print(f"Training rows: {len(X_train)} (Through 2023)")
print(f"Testing rows: {len(X_test)} (2024 onwards)")

In [ ]:
#Fit OLS model
X_train_const = sm.add_constant(X_train, has_constant='add')
X_test_const = sm.add_constant(X_test, has_constant='add')

ols_model = sm.OLS(y_train, X_train_const).fit()

# Predict and evaluate
y_pred = ols_model.predict(X_test_const)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:,.2f}")
print(f"MAE:  {mae:,.2f}")
print(f"R²:   {r2:.4f}")

# Full statistical summary
print(ols_model.summary())

In [ ]:
#Trying because Huber Regressor treats small errors as squared (like linear regression) but treats large errors (outliers) as linear so it's more robust to outliers
# 1. Create the Robust Pipeline
# epsilon: A smaller epsilon (closer to 1.0) makes it MORE robust to outliers.
# The default is 1.35, which is usually a good balance.
huber_pipeline = make_pipeline(
    StandardScaler(),
    HuberRegressor(epsilon=1.35, max_iter=1000)
)

# 2. Train the model
huber_pipeline.fit(X_train, y_train)

# 3. Predict
y_pred_huber = huber_pipeline.predict(X_test)

# 4. Evaluate
print(f"Huber RMSE: {mean_squared_error(y_test, y_pred_huber)**0.5:,.2f}")
print(f"Huber R²:   {r2_score(y_test, y_pred_huber):.4f}")

In [ ]:
#Try decision tree
# Initialize and fit
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)

# Predict and evaluate
y_pred_dt = dt_model.predict(X_test)

print(f"Decision Tree RMSE: {mean_squared_error(y_test, y_pred_dt)**0.5:,.2f}")
print(f"Decision Tree R²:   {r2_score(y_test, y_pred_dt):.4f}")

In [ ]:
#Try random forest
# Initialize and fit
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)

# Predict and evaluate
y_pred_rf = rf_model.predict(X_test)

print(f"Random Forest RMSE: {mean_squared_error(y_test, y_pred_rf)**0.5:,.2f}")
print(f"Random Forest R²:   {r2_score(y_test, y_pred_rf):.4f}")

In [ ]:
# 1. Initialize the TimeSeriesSplit (5 folds as requested by rubric)
tscv = TimeSeriesSplit(n_splits=5)

# 2. Calculate cross-validated R2 scores
for model in [ols_model, huber_pipeline, dt_model]:
    cv_scores = cross_val_score(dt_model, X_train, y_train, cv=tscv, scoring='r2')
    mean_r2 = np.mean(cv_scores)
    std_r2 = np.std(cv_scores)
    print(f"Mean R² for {model}: {mean_r2:.4f}")
    print(f"Std Dev R² for {model}: ±{std_r2:.4f}"

In [ ]:
plt.figure(figsize=(25,10))
plot_tree(dt_model,
          feature_names=X_train.columns,
          max_depth=3,
          filled=False,        # Removes the orange background
          impurity=False,     # Removes the "squared_error" text to declutter
          fontsize=12,        # Increases text size
          precision=2)        # Rounds numbers to 2 decimals
plt.show()

In [ ]:
#Evaluate feature importance
importances = dt_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print("Top Features:\n", feature_importance_df.head(10))

#Visualize results
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'][:10], feature_importance_df['Importance'][:10])
plt.gca().invert_yaxis()
plt.title("Top 10 Feature Importances (Decision Tree)")
plt.show()

In [ ]:
def ablation_test(list_feats):
    for feat in list_feats:
        X_train_ablated = X_train.drop(columns=[feat])
        X_test_ablated = X_test.drop(columns=[feat])

        dt_ablated = DecisionTreeRegressor(max_depth=10, random_state=42)
        dt_ablated.fit(X_train_ablated, y_train)
        y_pred_ablated = dt_ablated.predict(X_test_ablated)
        print(f"Ablated ({feat} removed) RMSE: {mean_squared_error(y_test, y_pred_ablated)**0.5:,.2f}")

ablation_test(['days_since_2000', 'longitude', 'downtown_core_district_within_mile', 'daytype_W', 'RED', 'transportation_district_within_half_mile',
               'heavy_industry_district_within_mile', 'latitude', 'ADA', 'BLUE'])

In [ ]:
# 1. Get the Baseline RMSE (for the "Zero Line" in the ablation chart)
baseline_rmse = mean_squared_error(y_test, dt_model.predict(X_test))**0.5

# 2. Modify function to return a dictionary of results
def get_ablation_data(list_feats):
    results = {}
    for feat in list_feats:
        X_train_ablated = X_train.drop(columns=[feat])
        X_test_ablated = X_test.drop(columns=[feat])

        dt_ablated = DecisionTreeRegressor(max_depth=10, random_state=42)
        dt_ablated.fit(X_train_ablated, y_train)
        y_pred_ablated = dt_ablated.predict(X_test_ablated)

        # Calculate the "Error Increase" (Ablated RMSE - Baseline RMSE)
        error_increase = (mean_squared_error(y_test, y_pred_ablated)**0.5) - baseline_rmse
        results[feat] = error_increase
    return results

# 3. Run the ablation and get the Gini Importance
top_10_feats = ['days_since_2000', 'longitude', 'downtown_core_district_within_mile', 'daytype_W', 'RED',
                'transportation_district_within_half_mile', 'heavy_industry_district_within_mile', 'latitude', 'ADA', 'BLUE']

ablation_dict = get_ablation_data(top_10_feats)
gini_importance = dict(zip(X.columns, dt_model.feature_importances_))

# 4. Combine into one DataFrame for plotting
df_comp = pd.DataFrame({
    'Gini Importance (0-1)': [gini_importance[f] for f in top_10_feats],
    'RMSE Increase when Removed': [ablation_dict[f] for f in top_10_feats]
}, index=top_10_feats).sort_values('Gini Importance (0-1)')

# 5. Visualize
fig, ax1 = plt.subplots(figsize=(12, 4.5))

# Plot Gini Importance as Bars
df_comp['Gini Importance (0-1)'].plot(kind='barh', ax=ax1, color='skyblue', label='Gini Importance', width=0.4, position=1)
ax1.set_xlabel('Gini Importance (Weight in Model)', color='blue')

# Create a second axis for Ablation (RMSE Increase)
ax2 = ax1.twiny()
df_comp['RMSE Increase when Removed'].plot(kind='barh', ax=ax2, label='Ablation (RMSE Increase)', width=0.4, position=0)
ax2.set_xlabel('RMSE Increase (Impact on Error)')

plt.title('Feature Comparison: Model Training Weight (Gini) vs. Prediction Impact (Ablation)', fontsize=14, pad=20)
ax1.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
depths = [2, 5, 7, 9, 10, 13, 15, 20, 30]
train_errors = []
test_errors = []

for d in depths:
    model = DecisionTreeRegressor(max_depth=d, random_state=42)
    model.fit(X_train, y_train)

    # Calculate RMSE for both sets
    train_rmse = mean_squared_error(y_train, model.predict(X_train))**0.5
    test_rmse = mean_squared_error(y_test, model.predict(X_test))**0.5

    train_errors.append(train_rmse)
    test_errors.append(test_rmse)

plt.plot(depths, train_errors, label='Train Error')
plt.plot(depths, test_errors, label='Test Error')
plt.xlabel('Max Depth')
plt.ylabel('RMSE')
plt.title('Sensitivity Analysis: Impact of Tree Depth')
plt.legend()
plt.show()

In [ ]:
# 1. Calculate errors and sort (same as before)
results = X_test.copy()
results['Actual'] = y_test
results['Predicted'] = y_pred_dt
results['Abs_Error'] = abs(results['Actual'] - results['Predicted'])

# 2. Get top 3 as a DataFrame
# We use .head(3) to keep it concise
top_failures_df = results.sort_values(by='Abs_Error', ascending=False).head(3)

# 3. Display the DataFrame
# If you are in a Jupyter Notebook, just typing the variable name prints a nice table
top_failures_df

In [ ]:
#The biggest miss was on March 15, 2025 (9205 days after 2000) at Grand (Red Line) and Clark/Lake (Transfer Station), followed by October 12, 2025 (9416 days) at Cermak-Chinatown (Red Line).
#Likely due to St. Patrick's Day for the former and the Chicago Marathon for the latter

In [ ]:
result = permutation_importance(
    rf_model, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1
)

# Organize the results
perm_sorted_idx = result.importances_mean.argsort()
perm_df = pd.DataFrame(
    result.importances[perm_sorted_idx].T,
    columns=X.columns[perm_sorted_idx]
)

# Plotting the result
plt.figure(figsize=(18, 8))
perm_df.iloc[:, -20:].boxplot(vert=False, manage_ticks=True)
plt.title("Permutation Importances (Test Set)")
plt.xlabel("Decrease in R² Score")
plt.tight_layout()
plt.show()

In [ ]:
# PCA on ridership_with_zoning (with fallback to ridershipo_with_zoning if that exists)
pca_source_df = ridership_with_zoning.copy()

# Drop non-feature columns that are not directly usable
pca_features = pca_source_df.drop(columns=['date', 'rides'], errors='ignore')

# Encode categorical fields
cat_cols = pca_features.select_dtypes(include=['object', 'string', 'category']).columns
pca_features = pd.get_dummies(pca_features, columns=cat_cols, drop_first=True, dtype=int)

# Convert bools and coerce to numeric
bool_cols = pca_features.select_dtypes(include='bool').columns
pca_features[bool_cols] = pca_features[bool_cols].astype(int)
pca_features = pca_features.apply(pd.to_numeric, errors='coerce').dropna()

# Optional sampling for speed
sample_n = min(200_000, len(pca_features))
pca_features = pca_features.sample(n=sample_n, random_state=42)

# Standardize then fit PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_features)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
cum_explained = np.cumsum(explained)
n95 = np.argmax(cum_explained >= 0.95) + 1

print(f"Rows used for PCA: {len(pca_features):,}")
print(f"Original feature count: {pca_features.shape[1]:,}")
print(f"Components to explain >=95% variance: {n95}")
print(f"First 10 explained variance ratios:\n{explained[:10]}")

# PCA scores dataframe (first n95 components)
si_pca_scores = pd.DataFrame(
    X_pca[:, :n95],
    columns=[f'PC{i+1}' for i in range(n95)],
    index=pca_features.index
)
display(si_pca_scores.head())


In [ ]:
cat_summary = ridership_with_zoning.select_dtypes(include=['object', 'string', 'category']).nunique().sort_values(ascending=False)
print(cat_summary)
print(f"\nTotal categorical features: {len(cat_summary)}")
print(f"Estimated expanded columns: {cat_summary.sum() - len(cat_summary)}")  # Rough estimate (subtract 1 per column for drop_first)

In [ ]:

# Scree + cumulative variance plots
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(range(1, len(explained) + 1), explained, marker='o', linewidth=1)
ax[0].set_title('Scree Plot')
ax[0].set_xlabel('Principal Component')
ax[0].set_ylabel('Explained Variance Ratio')

ax[1].plot(range(1, len(cum_explained) + 1), cum_explained, marker='o', linewidth=1)
ax[1].axhline(0.95, color='red', linestyle='--', linewidth=1)
ax[1].axvline(n95, color='green', linestyle='--', linewidth=1)
ax[1].set_title('Cumulative Explained Variance')
ax[1].set_xlabel('Principal Component')
ax[1].set_ylabel('Cumulative Variance Ratio')

plt.tight_layout()
plt.show()

In [ ]:
feature_names = pca_features.columns.copy()

print(f"Total features before PCA: {len(feature_names)}")

# Loadings matrix: rows=original features, cols=PCs
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_names,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)
# Remove station name features from loadings to declutter interpretation
station_name_mask = (
    loadings.index.str.startswith("stationname_") |
    loadings.index.str.startswith("STATION_NAME_")
)
loadings = loadings.loc[~station_name_mask].copy()
# Show top contributors for first 3 PCs
for pc in ["PC1", "PC2", "PC3"]:
    print(f"\nTop + contributors to {pc}")
    print(loadings[pc].sort_values(ascending=False).head(10))
    print(f"\nTop - contributors to {pc}")
    print(loadings[pc].sort_values(ascending=True).head(10))

# Contribution strength (absolute loading) for PC1/PC2/PC3
pc1_contrib = loadings[["PC1",]].abs().sort_values("PC1", ascending=False)
pc1_neg_contrib = loadings[["PC1",]].sort_values("PC1", ascending=True)

pc2_contrib = loadings[["PC2",]].abs().sort_values("PC2", ascending=False)
pc2_neg_contrib = loadings[["PC2",]].sort_values("PC2", ascending=True)

pc3_contrib = loadings[["PC3",]].abs().sort_values("PC3", ascending=False)
pc3_neg_contrib = loadings[["PC3",]].sort_values("PC3", ascending=True)

display(pc1_contrib.head(20))
display(pc1_neg_contrib.head(20))

display(pc2_contrib.head(20))
display(pc2_neg_contrib.head(20))

display(pc3_contrib.head(20))
display(pc3_neg_contrib.head(20))

In [ ]:
def get_top_contributors(loadings_df, pc_count, top_n=10):
    top_features = set()
    for i in range(1, pc_count + 1):
        pc_col = f"PC{i}"
        top_features.update(loadings_df[pc_col].abs().sort_values(ascending=False).head(top_n).index)
    return list(top_features)

result = get_top_contributors(loadings, pc_count=3, top_n=10)
result

In [ ]:
n_features_in_loadings = loadings.shape[0]
max_len_get_top_contributors = len(
    get_top_contributors(loadings, pc_count=loadings.shape[1], top_n=n_features_in_loadings)
)

print(f"Number of features in `loadings after station mask`: {n_features_in_loadings}")
print(f"Max length from `get_top_contributors`: {max_len_get_top_contributors}")

In [ ]:
# Add PCA columns to ridership_with_zoning
# Prepare features for PCA (same process as before)
pca_prep_df = ridership_with_zoning.copy()
cat_cols_pca = pca_prep_df.select_dtypes(include=['object', 'string', 'category']).columns
pca_prep_df = pd.get_dummies(pca_prep_df, columns=cat_cols_pca, drop_first=True, dtype=int)

# Convert bools to int and coerce to numeric
bool_cols_pca = pca_prep_df.select_dtypes(include='bool').columns
pca_prep_df[bool_cols_pca] = pca_prep_df[bool_cols_pca].astype(int)
pca_prep_df = pca_prep_df.apply(pd.to_numeric, errors='coerce')

# Drop rows with any NaN values for PCA fitting
pca_prep_df_clean = pca_prep_df.dropna()

# Standardize and fit PCA
scaler_final = StandardScaler()
X_scaled_final = scaler_final.fit_transform(pca_prep_df_clean)

pca_final = PCA()
X_pca_final = pca_final.fit_transform(X_scaled_final)

# Use 95% variance threshold
cum_var_final = np.cumsum(pca_final.explained_variance_ratio_)
n_components_95 = np.argmax(cum_var_final >= 0.95) + 1

print(f"Number of components needed for 95% variance: {n_components_95}")
print(f"Total rows used for PCA: {len(pca_prep_df_clean):,}")

# Create dataframe with PC columns
pca_columns_df = pd.DataFrame(
    X_pca_final[:, :n_components_95],
    columns=[f'PC{i+1}' for i in range(n_components_95)],
    index=pca_prep_df_clean.index
)

# Add PCA columns to ridership_with_zoning (aligned by index)
ridership_with_zoning_pca = ridership_with_zoning.copy()
ridership_with_zoning_pca = ridership_with_zoning_pca.loc[pca_columns_df.index].copy()
ridership_with_zoning_pca = pd.concat([ridership_with_zoning_pca.reset_index(drop=True), 
                                        pca_columns_df.reset_index(drop=True)], axis=1)

print(f"\nOriginal ridership_with_zoning shape: {ridership_with_zoning.shape}")
print(f"ridership_with_zoning_pca shape: {ridership_with_zoning_pca.shape}")
print(f"New PC columns: {list(ridership_with_zoning_pca.columns[-n_components_95:])}")

display(ridership_with_zoning_pca.head())

In [ ]:
def get_pc_cols(n):
    return [f"PC{i+1}" for i in range(n)]

# Build PCA-only modeling frame
pc_cols = sorted(
    [c for c in ridership_with_zoning_pca.columns if re.fullmatch(r"PC\d+", c)],
    key=lambda x: int(x[2:])
)

pc_model_df = ridership_with_zoning_pca[['days_since_2000', 'rides'] + pc_cols].copy()
pc_model_df = pc_model_df.apply(pd.to_numeric, errors='coerce').dropna()

X_pc = pc_model_df[['days_since_2000'] + pc_cols]
y_pc = pc_model_df['rides']

# Time-based split (reuse existing cutoff if present)
cutoff_value_pc = cutoff_value if 'cutoff_value' in globals() else 25 * 365.25
train_mask_pc = X_pc['days_since_2000'] <= cutoff_value_pc
test_mask_pc = X_pc['days_since_2000'] > cutoff_value_pc

X_train_pc, X_test_pc = X_pc.loc[train_mask_pc], X_pc.loc[test_mask_pc]
y_train_pc, y_test_pc = y_pc.loc[train_mask_pc], y_pc.loc[test_mask_pc]

print(f"PC train rows: {len(X_train_pc)}")
print(f"PC test rows:  {len(X_test_pc)}")
print(f"Available PC columns: {len(pc_cols)}")

# Evaluate different numbers of PCs
results = []
for count in range(1,10):
    cols = get_pc_cols(count)
    X_train_subset = X_train_pc[cols]
    X_test_subset = X_test_pc[cols]

    model = DecisionTreeRegressor(max_depth=10, random_state=42)
    model.fit(X_train_subset, y_train_pc)
    y_pred_subset = model.predict(X_test_subset)

    rmse_subset = mean_squared_error(y_test_pc, y_pred_subset) ** 0.5
    r2_subset = r2_score(y_test_pc, y_pred_subset)
    results.append((count, rmse_subset, r2_subset))
# Display results
results_df = pd.DataFrame(results, columns=["PC Count", "RMSE", "R2"])
print(results_df)

In [ ]:
# Dual plots: R² and RMSE vs PC Count
if 'results_df' in globals() and {'PC Count', 'RMSE', 'R2'}.issubset(results_df.columns):
    plot_df = results_df.copy()
elif 'results' in globals():
    # Fallback if tuple-list exists: (PC Count, RMSE, R2)
    plot_df = pd.DataFrame(results, columns=['PC Count', 'RMSE', 'R2'])
else:
    raise ValueError("No PCA evaluation results found. Run the PCA model comparison cell first.")

plot_df = plot_df.sort_values('PC Count').reset_index(drop=True)

best_r2_idx = plot_df['R2'].idxmax()
best_rmse_idx = plot_df['RMSE'].idxmin()

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)

# R² plot
sns.lineplot(data=plot_df, x='PC Count', y='R2', marker='o', ax=axes[0])
axes[0].scatter(
    plot_df.loc[best_r2_idx, 'PC Count'],
    plot_df.loc[best_r2_idx, 'R2'],
    color='red',
    zorder=5,
    label='Best R²'
)
axes[0].set_title('R² vs PC Count')
axes[0].set_xlabel('PC Count')
axes[0].set_ylabel('R²')
axes[0].grid(alpha=0.3)
axes[0].legend()

# RMSE plot
sns.lineplot(data=plot_df, x='PC Count', y='RMSE', marker='o', ax=axes[1], color='orange')
axes[1].scatter(
    plot_df.loc[best_rmse_idx, 'PC Count'],
    plot_df.loc[best_rmse_idx, 'RMSE'],
    color='red',
    zorder=5,
    label='Best RMSE'
)
axes[1].set_title('RMSE vs PC Count')
axes[1].set_xlabel('PC Count')
axes[1].set_ylabel('RMSE')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Best by R²   -> PC Count: {int(plot_df.loc[best_r2_idx, 'PC Count'])}, R²: {plot_df.loc[best_r2_idx, 'R2']:.4f}")
print(f"Best by RMSE -> PC Count: {int(plot_df.loc[best_rmse_idx, 'PC Count'])}, RMSE: {plot_df.loc[best_rmse_idx, 'RMSE']:.2f}")

By using principal components to predict ridership, performance seemed to peak using 4 principal components. Let's see if we can use the 3 principal components to feature engineer a lighter model that uses the top contributing features for all 4 principal coompents

In [ ]:
plot_data = []
light_ridership_df = ridership_with_zoning.copy()

# print(ridership_with_zoning.columns)

# Iterate through different numbers of top features to evaluate model performance
for top_features_count in range(10, 100, 10):
    top_features = get_top_contributors(loadings, pc_count=3, top_n=top_features_count)
    
    light_ridership_df = ridership_with_zoning.copy()
    cat_cols = light_ridership_df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    light_ridership_df = pd.get_dummies(light_ridership_df, columns=cat_cols, drop_first=True, dtype=int)

    # Keep only available columns and prevent duplicates (especially days_since_2000)
    selected_cols = [c for c in (top_features + ['rides', 'days_since_2000']) if c in light_ridership_df.columns]
    selected_cols = list(dict.fromkeys(selected_cols))
    light_ridership_df = light_ridership_df.loc[:, selected_cols].dropna()
    light_ridership_df = light_ridership_df.loc[:, ~light_ridership_df.columns.duplicated()]
    
    X_light = light_ridership_df.drop(columns=['rides'])
    y_light = light_ridership_df['rides']
    
    # Time-based split
    train_mask_l = X_light['days_since_2000'] <= cutoff_value
    test_mask_l = X_light['days_since_2000'] > cutoff_value
    
    X_train_l, X_test_l = X_light.loc[train_mask_l], X_light.loc[test_mask_l]
    y_train_l, y_test_l = y_light.loc[train_mask_l], y_light.loc[test_mask_l]
    
    model_l = DecisionTreeRegressor(max_depth=10, random_state=42)
    model_l.fit(X_train_l, y_train_l)
    y_pred_l = model_l.predict(X_test_l)
    rmse_l = mean_squared_error(y_test_l, y_pred_l)**0.5
    r2_l = r2_score(y_test_l, y_pred_l)
    print(f"PCs: 3, Top N: {top_features_count} => RMSE: {rmse_l:,.2f}, R²: {r2_l:.4f}")
    plot_data.append({
        'Top_features': top_features_count,
        'RMSE': rmse_l,
        'R2': r2_l
    })

In [ ]:
plot_df = pd.DataFrame(plot_data)
plt.figure(figsize=(12, 6))
sns.lineplot(data=plot_df, x='Top_features', y='R2', marker='o')
plt.title('R² vs Number of Top Contributing Features within First 4 PCs')
plt.xlabel('Number of Top Contributing Features')
plt.ylabel('R² Score')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
display(get_top_contributors(loadings, pc_count=3, top_n=80))

unused_features = set(X.columns) - set(get_top_contributors(loadings, pc_count=3, top_n=80))
print(f"Number of features not in top 80 contributors: {len(unused_features)}")
print(f"Examples of features not in top 80 contributors: {list(unused_features)}")

It turns out you still need most of the top contributing features for the 3 principal components for good model performance. In this situation, using the top contributing PC features is probably not a good way to feature engineer a lighter model for ridership.

In [ ]:
start_year = 2018
end_year = pd.Timestamp.today().year

#Get average seasonal and annual ridership
si_gdf_avg_ridership = si_gdf.copy()

for idx, station in si_gdf_avg_ridership.iterrows():
    map_id = station['MAP_ID']

    station_slice = station_ridership[
        (station_ridership['MAP_ID'] == map_id) &
        (station_ridership['year'] >= start_year) &
        (station_ridership['year'] <= end_year)
    ]

    seasonal_avg = (
        station_slice
        .groupby(['year', 'season'], as_index=False)['rides']
        .mean()
        .rename(columns={'rides': 'avg_rides'})
        .sort_values(['year', 'season'])
    )

    for _, season_row in seasonal_avg.iterrows():
        column_name = f"{int(season_row['year'])}_{season_row['season']}_avg_rides"
        si_gdf_avg_ridership.at[idx, column_name] = season_row['avg_rides']

    annual_avg = (
        station_slice
        .groupby('year', as_index=False)['rides']
        .mean()
        .rename(columns={'rides': 'avg_rides_annual'})
        .sort_values('year')
    )

    for _, year_row in annual_avg.iterrows():
        column_name = f"{int(year_row['year'])}_avg_rides"
        si_gdf_avg_ridership.at[idx, column_name] = year_row['avg_rides_annual']

for column in si_gdf_avg_ridership:
    print(column)


In [ ]:

# Use all ridership-average features created in si_gdf_avg_ridership
exclude_cols = {"MAP_ID", "STATION_NAME", "stationname"}
clustering_cols = [
    c for c in si_gdf_avg_ridership.columns
    if c not in exclude_cols and not c.startswith("geometry")
]

if not clustering_cols:
    raise ValueError("No *_avg_rides columns found in si_gdf_avg_ridership.")

# Build clustering matrix (median-impute missing values)
X_cluster = si_gdf_avg_ridership[clustering_cols].copy()
X_cluster = X_cluster.apply(pd.to_numeric, errors="coerce")
X_cluster = X_cluster.fillna(X_cluster.median())

# Scale + cluster
X_scaled = StandardScaler().fit_transform(X_cluster)

k = 5  # adjust if needed
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

si_gdf_avg_ridership_k5 = si_gdf_avg_ridership.copy()

si_gdf_avg_ridership_k5["cluster"] = kmeans.fit_predict(X_scaled)

si_gdf_avg_ridership_k5.head()
 

In [ ]:

# Quick summary
# Average ridership statistics by cluster
ridership_cols = [c for c in si_gdf_avg_ridership.columns if c.endswith("_avg_rides")]

if ridership_cols:
    cluster_ridership_stats = (
        si_gdf_avg_ridership_k5
        .assign(_station_avg_rides=si_gdf_avg_ridership_k5[ridership_cols].mean(axis=1, skipna=True))
        .groupby("cluster")["_station_avg_rides"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .rename(columns={"count": "n_stations"})
        .round(2)
    )
    print("\nAverage ridership statistics by cluster:")
    display(cluster_ridership_stats)
else:
    print("No *_avg_rides columns found for ridership statistics.")

# Plot clusters by station location
plt.figure(figsize=(10, 8))
label_col = "STATION_NAME" if "STATION_NAME" in si_gdf_avg_ridership_k5.columns else "stationname"
si_gdf_avg_ridership_cluster_ = si_gdf_avg_ridership_k5[si_gdf_avg_ridership_k5["cluster"] == 0].copy()

si_gdf_avg_ridership_cluster_0 = si_gdf_avg_ridership_k5[si_gdf_avg_ridership_k5["cluster"] == 0].copy()

for _, row in si_gdf_avg_ridership_cluster_0.dropna(subset=["longitude", "latitude", [label_col][0]]).iterrows():
    plt.text(
        row["longitude"] + 0.002,
        row["latitude"] + 0.002,
        str(row[label_col]),
        fontsize=7,
        alpha=0.85
    )
sns.scatterplot(
    data=si_gdf_avg_ridership_k5,
    x="longitude",
    y="latitude",
    hue="cluster",
    palette="tab10",
    s=70,
    alpha=0.9
)
plt.title("Station Clusters (based on all features)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

We can see clear clusters. Cluster 0 seems to be high traffic. centered around the Loop, O'hare, Fullerton (a commonly used transfer stop between the red and brown line) and Belmont (close to Wrigleyville and the famous Wrigley Stadium, home of the Chicago Cubs baseball team). Cluster 1 and 4 are medium traffic stations with stations in Cluster 1 being closer to the Loop on average than stations in cluster 4. Clusters 2 and 3 are lower traffic stations with stations in Cluster 2 on average being closer to the Loop than those in Cluster 3.

In [ ]:
# Ablation testing to see which features are most important for clustering
from sklearn.metrics import silhouette_score
feature_importance = {}

# Calculate baseline silhouette score
baseline_score = silhouette_score(X_scaled, si_gdf_avg_ridership_k5["cluster"])
print(f"Baseline silhouette score: {baseline_score:.4f}\n")

for col in clustering_cols:
    X_ablated = X_cluster.drop(columns=[col])
    X_ablated_scaled = StandardScaler().fit_transform(X_ablated)
    kmeans_ablated = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_ablated = kmeans_ablated.fit_predict(X_ablated_scaled)
    score = silhouette_score(X_ablated_scaled, labels_ablated)
    feature_importance[col] = score
feature_importance_df = pd.DataFrame.from_dict(feature_importance, orient="index", columns=["silhouette_score"])
feature_importance_df = feature_importance_df.sort_values(by="silhouette_score", ascending=True)
print("Most important features based on silhouette score after ablation (lower score indicates higher importance):")
display(feature_importance_df)

- It looks like each individual feature using clustering doesn't affect the sillhouette score that much, indicating redundant features or weak clusters. The sillhouette score of the unablated model is only 0.24, which isn't the strongest clustering. Let's try again with fewer clusters

In [ ]:
# Retry clustering with k=3 for stronger, more distinct clusters
k = 3

# Work with a copy
si_gdf_avg_ridership_k3 = si_gdf_avg_ridership.copy()

kmeans_3 = KMeans(n_clusters=k, random_state=42, n_init=10)
si_gdf_avg_ridership_k3["cluster"] = kmeans_3.fit_predict(X_scaled)

# Calculate silhouette score
silhouette_3 = silhouette_score(X_scaled, si_gdf_avg_ridership_k3["cluster"])
print(f"Silhouette score with {k} clusters: {silhouette_3:.4f}\n")

# Average ridership statistics by cluster
ridership_cols = [c for c in si_gdf_avg_ridership_k3.columns if c.endswith("_avg_rides")]

if ridership_cols:
    cluster_ridership_stats = (
        si_gdf_avg_ridership_k3
        .assign(_station_avg_rides=si_gdf_avg_ridership_k3[ridership_cols].mean(axis=1, skipna=True))
        .groupby("cluster")["_station_avg_rides"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .rename(columns={"count": "n_stations"})
        .round(2)
    )
    print("Average ridership statistics by cluster:")
    display(cluster_ridership_stats)

# Plot clusters by station location
plt.figure(figsize=(10, 8))
label_col = "STATION_NAME" if "STATION_NAME" in si_gdf_avg_ridership_k3.columns else "stationname"

for _, row in si_gdf_avg_ridership_k3.dropna(subset=["longitude", "latitude", label_col]).iterrows():
    plt.text(
        row["longitude"] + 0.002,
        row["latitude"] + 0.002,
        str(row[label_col]),
        fontsize=7,
        alpha=0.85
    )
sns.scatterplot(
    data=si_gdf_avg_ridership_k3,
    x="longitude",
    y="latitude",
    hue="cluster",
    palette="Set1",
    s=70,
    alpha=0.9
)
plt.title(f"Station Clusters (k={k})")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

Fewer clusters show moderate improvements to sillhouette score. Tradeoff of clearer clusters for nuance

In [ ]:
# Ablation test for the k=3 clustering setup (same pattern as before)
feature_importance_k3 = {}

# Ensure k is aligned with the 3-cluster dataframe
k = 3

# Baseline silhouette score using existing scaled matrix and k=3 labels
baseline_score_k3 = silhouette_score(X_scaled, si_gdf_avg_ridership_k3["cluster"])
print(f"Baseline silhouette score (k=3): {baseline_score_k3:.4f}\n")

for col in clustering_cols:
    X_ablated = X_cluster.drop(columns=[col])
    X_ablated_scaled = StandardScaler().fit_transform(X_ablated)
    
    kmeans_ablated = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_ablated = kmeans_ablated.fit_predict(X_ablated_scaled)
    
    score = silhouette_score(X_ablated_scaled, labels_ablated)
    feature_importance_k3[col] = score

feature_importance_k3_df = (
    pd.DataFrame.from_dict(feature_importance_k3, orient="index", columns=["silhouette_score"])
    .assign(delta_vs_baseline=lambda d: d["silhouette_score"] - baseline_score_k3)
    .sort_values(by="silhouette_score", ascending=True)
)

print("Most important features for k=3 ablation (lower silhouette after ablation = more important):")
display(feature_importance_k3_df)

Let's try to join cluster labels and see if they improve the model or not

In [ ]:
ridership_with_zoning_k5 = ridership_with_zoning.copy()
cluster_map = dict(zip(si_gdf_avg_ridership_k5["MAP_ID"], si_gdf_avg_ridership_k5["cluster"]))

cluster_map
ridership_with_zoning_k5["cluster_k5"] = ridership_with_zoning_k5["MAP_ID"].map(cluster_map)

# Check how many rows got a cluster assignment
assigned_clusters = ridership_with_zoning_k5["cluster_k5"].notna().sum()
total_rows = len(ridership_with_zoning_k5)
print(f"Rows with cluster assignment: {assigned_clusters} / {total_rows} ({assigned_clusters/total_rows:.2%})")

# show rows with no clusters
no_cluster_rows = ridership_with_zoning_k5[ridership_with_zoning_k5["cluster_k5"].isna()]
print(f"Rows without cluster assignment: {len(no_cluster_rows)}")
display(no_cluster_rows[['stationname', 'MAP_ID', 'rides', 'cluster_k5']])

display(ridership_with_zoning_k5.head())

In [ ]:
model_k5 = ridership_with_zoning_k5.copy()

base_features_k5 = [
    'days_since_2000', 'post_shutdown', 'season_Spring', 'season_Summer', 'season_Winter',
    'year', 'PRCP', 'TAVG', 'SNOW', 'SNWD', 'Fog_Ice_Fog', 'Blowing_Snow',
    'latitude', 'longitude', 'ADA', 'RED', 'BLUE', 'G', 'BRN', 'P', 'Y', 'Pnk', 'O', 'N/S', 'E/W'
]
zoning_features_k5 = [c for c in model_k5.columns if c.endswith('_mile') or c.endswith('_half_mile')]
feature_cols_k5 = [c for c in base_features_k5 if c in model_k5.columns] + zoning_features_k5 + ['cluster_k5']

X_k5 = model_k5[feature_cols_k5].copy()
y_k5 = pd.to_numeric(model_k5['rides'], errors='coerce')

# Encode daytype + cluster label
daytype_dummies_k5 = pd.get_dummies(model_k5['daytype'], prefix='daytype', drop_first=True, dtype=int)
cluster_dummies_k5 = pd.get_dummies(model_k5['cluster_k5'], prefix='cluster_k5', drop_first=True, dtype=int)

X_k5 = pd.concat([X_k5.drop(columns=['cluster_k5']), daytype_dummies_k5, cluster_dummies_k5], axis=1)

# Ensure numeric
bool_cols_k5 = X_k5.select_dtypes(include='bool').columns
X_k5[bool_cols_k5] = X_k5[bool_cols_k5].astype(int)
X_k5 = X_k5.apply(pd.to_numeric, errors='coerce')

valid_mask_k5 = X_k5.notna().all(axis=1) & y_k5.notna()
X_k5 = X_k5.loc[valid_mask_k5]
y_k5 = y_k5.loc[valid_mask_k5]

# Time-series split
cutoff_value_k5 = cutoff_value if 'cutoff_value' in globals() else 25 * 365.25
train_mask_k5 = X_k5['days_since_2000'] <= cutoff_value_k5
test_mask_k5 = X_k5['days_since_2000'] > cutoff_value_k5

X_train_k5, X_test_k5 = X_k5.loc[train_mask_k5], X_k5.loc[test_mask_k5]
y_train_k5, y_test_k5 = y_k5.loc[train_mask_k5], y_k5.loc[test_mask_k5]

print(f"Train rows: {len(X_train_k5):,}")
print(f"Test rows:  {len(X_test_k5):,}")

# Decision Tree model
dt_k5 = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_k5.fit(X_train_k5, y_train_k5)

y_pred_k5 = dt_k5.predict(X_test_k5)

rmse_k5 = mean_squared_error(y_test_k5, y_pred_k5) ** 0.5
mae_k5 = mean_absolute_error(y_test_k5, y_pred_k5)
r2_k5 = r2_score(y_test_k5, y_pred_k5)

print(f"Decision Tree (with k5 clusters) RMSE: {rmse_k5:,.2f}")
print(f"Decision Tree (with k5 clusters) MAE:  {mae_k5:,.2f}")
print(f"Decision Tree (with k5 clusters) R²:   {r2_k5:.4f}")

In [ ]:
ridership_with_zoning_k3 = ridership_with_zoning.copy()
cluster_map = dict(zip(si_gdf_avg_ridership_k3["MAP_ID"], si_gdf_avg_ridership_k3["cluster"]))

cluster_map
ridership_with_zoning_k3["cluster_k3"] = ridership_with_zoning_k3["MAP_ID"].map(cluster_map)

# Check how many rows got a cluster assignment
assigned_clusters = ridership_with_zoning_k3["cluster_k3"].notna().sum()
total_rows = len(ridership_with_zoning_k3)
print(f"Rows with cluster assignment: {assigned_clusters} / {total_rows} ({assigned_clusters/total_rows:.2%})")

# show rows with no clusters
no_cluster_rows = ridership_with_zoning_k3[ridership_with_zoning_k3["cluster_k3"].isna()]
print(f"Rows without cluster assignment: {len(no_cluster_rows)}")
display(no_cluster_rows[['stationname', 'MAP_ID', 'rides', 'cluster_k3']])

display(ridership_with_zoning_k3.head())

In [ ]:
model_k3 = ridership_with_zoning_k3.copy()

base_features_k3 = [
    'days_since_2000', 'post_shutdown', 'season_Spring', 'season_Summer', 'season_Winter',
    'year', 'PRCP', 'TAVG', 'SNOW', 'SNWD', 'Fog_Ice_Fog', 'Blowing_Snow',
    'latitude', 'longitude', 'ADA', 'RED', 'BLUE', 'G', 'BRN', 'P', 'Y', 'Pnk', 'O', 'N/S', 'E/W'
]
zoning_features_k3 = [c for c in model_k3.columns if c.endswith('_mile') or c.endswith('_half_mile')]
feature_cols_k3 = [c for c in base_features_k3 if c in model_k3.columns] + zoning_features_k3 + ['cluster_k3']

X_k3 = model_k3[feature_cols_k3].copy()
y_k3 = pd.to_numeric(model_k3['rides'], errors='coerce')

# Encode daytype + cluster label
daytype_dummies_k3 = pd.get_dummies(model_k3['daytype'], prefix='daytype', drop_first=True, dtype=int)
cluster_dummies_k3 = pd.get_dummies(model_k3['cluster_k3'], prefix='cluster_k3', drop_first=True, dtype=int)

X_k3 = pd.concat([X_k3.drop(columns=['cluster_k3']), daytype_dummies_k3, cluster_dummies_k3], axis=1)

# Ensure numeric
bool_cols_k3 = X_k3.select_dtypes(include='bool').columns
X_k3[bool_cols_k3] = X_k3[bool_cols_k3].astype(int)
X_k3 = X_k3.apply(pd.to_numeric, errors='coerce')

valid_mask_k3 = X_k3.notna().all(axis=1) & y_k3.notna()
X_k3 = X_k3.loc[valid_mask_k3]
y_k3 = y_k3.loc[valid_mask_k3]

# Time-series split
cutoff_value_k3 = cutoff_value if 'cutoff_value' in globals() else 25 * 365.25
train_mask_k3 = X_k3['days_since_2000'] <= cutoff_value_k3
test_mask_k3 = X_k3['days_since_2000'] > cutoff_value_k3

X_train_k3, X_test_k3 = X_k3.loc[train_mask_k3], X_k3.loc[test_mask_k3]
y_train_k3, y_test_k3 = y_k3.loc[train_mask_k3], y_k3.loc[test_mask_k3]

print(f"Train rows: {len(X_train_k3):,}")
print(f"Test rows:  {len(X_test_k3):,}")

# Decision Tree model
dt_k3 = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_k3.fit(X_train_k3, y_train_k3)

y_pred_k3 = dt_k3.predict(X_test_k3)

rmse_k3 = mean_squared_error(y_test_k3, y_pred_k3) ** 0.5
mae_k3 = mean_absolute_error(y_test_k3, y_pred_k3)
r2_k3 = r2_score(y_test_k3, y_pred_k3)

print(f"Decision Tree (with k3 clusters) RMSE: {rmse_k3:,.2f}")
print(f"Decision Tree (with k3 clusters) MAE:  {mae_k3:,.2f}")
print(f"Decision Tree (with k3 clusters) R²:   {r2_k3:.4f}")

BASELINE DECISION TREE:
RMSE: 826.03
R²:   0.8464

DECISION TREE RUN PURELY ON 3 PRINCIPAL COMPONENTS
RMSE: 822.969281  
R²: 0.846626

DECISION TREE ON TOP 80 NON-STATION NAME CONTRIBUTORS FOR TOP 3 PRINCIPAL COMPONENTS
RMSE: 815.14, 
R²: 0.8495

DECISION TREE WITH K5 clusters
Decision Tree (with k5 clusters) RMSE: 891.61
Decision Tree (with k5 clusters) R²:   0.8210

DECISION TREE WITH K3 clusters
Decision Tree (with k3 clusters) RMSE: 824.38
Decision Tree (with k3 clusters) R²:   0.8470